In [2]:
import numpy as np
import pandas as pd
import joblib
import json
import time

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

DATA_DIR = Path("../data/raw")

train_df = pd.read_csv(DATA_DIR / "UNSW_NB15_training-set.csv")
test_df = pd.read_csv(DATA_DIR / "UNSW_NB15_testing-set.csv")

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

Training data: (175341, 45)
Testing data: (82332, 45)


In [6]:
excluded_columns = ["id", "label", "attack_cat"]

X_train = train_df.drop(columns=excluded_columns).copy()
y_train = train_df["label"].astype(int)

X_test = test_df.drop(columns=excluded_columns).copy()
y_test = test_df["label"].astype(int)

# Keep the test feature order identical to training
X_test = X_test[X_train.columns]

categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = [
    col for col in X_train.columns
    if col not in categorical_columns
]

print("Features:", X_train.shape[1])
print("Numerical features:", len(numerical_columns))
print("Categorical features:", categorical_columns)
print("Training labels:", y_train.value_counts().to_dict())
print("Test labels:", y_test.value_counts().to_dict())

Features: 42
Numerical features: 39
Categorical features: ['proto', 'service', 'state']
Training labels: {1: 119341, 0: 56000}
Test labels: {1: 45332, 0: 37000}


C:\Users\Nitin\AppData\Local\Temp\ipykernel_19012\192271850.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [7]:
def build_preprocessor():
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

def build_model(classifier):
    return Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", classifier)
    ])

models = {
    "Random Forest": build_model(
        RandomForestClassifier(
            n_estimators=150,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),
    "XGBoost": build_model(
        XGBClassifier(
            n_estimators=250,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
}

In [8]:
training_times = {}

for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()

    model.fit(X_train, y_train)

    training_times[name] = round(time.time() - start, 2)
    print(f"{name} trained in {training_times[name]} seconds.")

Training Random Forest...
Random Forest trained in 91.23 seconds.
Training XGBoost...
XGBoost trained in 6.85 seconds.


In [9]:
thresholds = {
    "Random Forest": 0.4,
    "XGBoost": 0.5,
    "Weighted Ensemble": 0.5
}

rf_probs = models["Random Forest"].predict_proba(X_test)[:, 1]
xgb_probs = models["XGBoost"].predict_proba(X_test)[:, 1]

probabilities_by_model = {
    "Random Forest": rf_probs,
    "XGBoost": xgb_probs,
    "Weighted Ensemble": 0.5 * rf_probs + 0.5 * xgb_probs
}

results = []
confusion_matrices = {}

for name, probabilities in probabilities_by_model.items():
    threshold = thresholds[name]
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, predictions),
        "Attack Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Attack Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "Attack F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "False Positive Rate": fp / (fp + tn),
        "False Negatives": fn,
        "False Positives": fp
    })

    confusion_matrices[name] = {
        "TN": int(tn), "FP": int(fp),
        "FN": int(fn), "TP": int(tp)
    }

    print(f"\n{name} — threshold {threshold}")
    print(classification_report(
        y_test, predictions,
        target_names=["Normal", "Attack"],
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions, labels=[0, 1]))

results_df = pd.DataFrame(results)
display(results_df.round(4))


Random Forest — threshold 0.4
              precision    recall  f1-score   support

      Normal       0.98      0.73      0.84     37000
      Attack       0.82      0.99      0.90     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.87      0.87     82332

Confusion matrix:
[[27122  9878]
 [  558 44774]]

XGBoost — threshold 0.5
              precision    recall  f1-score   support

      Normal       0.97      0.74      0.84     37000
      Attack       0.82      0.98      0.90     45332

    accuracy                           0.88     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.88      0.87     82332

Confusion matrix:
[[27465  9535]
 [  717 44615]]

Weighted Ensemble — threshold 0.5
              precision    recall  f1-score   support

      Normal       0.97      0.77      0.86     37000
      Attack       0.84      0.98      0.90     

,Model,Threshold,Accuracy,Balanced Accuracy,Attack Precision,Attack Recall,Attack F1,ROC-AUC,False Positive Rate,False Negatives,False Positives
0,Random Forest,0.4,0.8732,0.8604,0.8193,0.9877,0.8956,0.9839,0.2670,558,9878
1,XGBoost,0.5,0.8755,0.8632,0.8239,0.9842,0.8969,0.9841,0.2577,717,9535
2,Weighted Ensemble,0.5,0.8866,0.8760,0.8401,0.9807,0.9049,0.9845,0.2288,876,8464


In [10]:
from pathlib import Path
import json
import joblib

models_dir = Path("models")
reports_dir = Path("reports")

models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Save both trained pipelines
joblib.dump(
    models["Random Forest"],
    models_dir / "random_forest.joblib"
)

joblib.dump(
    models["XGBoost"],
    models_dir / "xgboost.joblib"
)

# Save ensemble settings
ensemble_config = {
    "rf_weight": 0.5,
    "xgb_weight": 0.5,
    "threshold": 0.5,
    "feature_columns": X_train.columns.tolist(),
    "positive_class": 1,
    "dataset": "UNSW-NB15"
}

with open(models_dir / "ensemble_config.json", "w") as f:
    json.dump(ensemble_config, f, indent=2)

# Save metrics and confusion matrices
results_df.to_csv(
    reports_dir / "final_test_metrics.csv",
    index=False
)

with open(reports_dir / "confusion_matrices.json", "w") as f:
    json.dump(confusion_matrices, f, indent=2)

print("Models and evaluation reports saved successfully.")

Models and evaluation reports saved successfully.


In [7]:
# from pathlib import Path

# sample = test_df.drop(
#     columns=["id", "label", "attack_cat"]
# ).head(20)

# output_path = Path("data/processed/demo_flows.csv")
# output_path.parent.mkdir(parents=True, exist_ok=True)

# sample.to_csv(output_path, index=False)

# print("Saved:", output_path.resolve())
# print("Rows:", len(sample))
# print("Features:", len(sample.columns))